# Kaggriculture | Adaptive Farm Intelligence — Fieldbook Exact

A compact Kaggriculture submission package.

## Method

An exact, self-contained reproduction of the public three-day Fieldbook route selector. It chooses one of six action programmes at the visible shop boundaries on days 6 and 9.

The notebook writes the required `submission.tar.gz`.

In [ ]:
# Exact submission archive with an integrity check.
import base64
import hashlib
import io
import tarfile
from pathlib import Path

ARCHIVE_B64 = (
    '...'
)
EXPECTED_SHA256 = "cd33673de33ef38937aba9bf6b94f71102fa6985f725fc1a718ed3202b14632b"
payload = base64.b64decode(ARCHIVE_B64)
assert hashlib.sha256(payload).hexdigest() == EXPECTED_SHA256
with tarfile.open(fileobj=io.BytesIO(payload), mode="r:gz") as archive:
    assert archive.getnames() == ["main.py"]
    source = archive.extractfile("main.py").read()
    compile(source, "main.py", "exec")
Path("submission.tar.gz").write_bytes(payload)
print("submission.tar.gz ready")
print("sha256:", EXPECTED_SHA256)
print("agent source:", len(source), "bytes")
